# Part 1: class summary by buffer

**Calculate total area and percent occupied by class for buffers of CWMP and GLFWRA bird count locations.**

_Buffers_: 200m, 500m, 1km, 2km  
_Data_: land cover and wetland type & code  
_Years_: 2010, 2015, 2020, 2023  
_Deliverable_: CSV file for each bird count dataset containing class summaries for each buffer for each location.

## Data

### Land Cover

##### [National Land Cover Database (NLCD)](https://www.usgs.gov/data/annual-national-land-cover-database-nlcd-collection-1-products-ver-11-june-2025)  
*source*: United States Geological Survey (USGS)  
*years*: 2010, 2015, 2020, 2023  
*metadata*: https://www.mrlc.gov/downloads/sciweb1/shared/mrlc/metadata/Annual_NLCD_LndCov_2024_CU_C1V1.xml
*how to cite*:  
U.S. Geological Survey (USGS), 2024, Annual NLCD Collection 1 Science Products (ver. 1.1, June 2025): U.S. Geological Survey data release, https://doi.org/10.5066/P94UXNTS.


##### [North American Land Change Monitoring System](https://www.cec.org/north-american-land-change-monitoring-system/)  
*source*: Commission for Environmental Cooperation (CEC)  
*years*: 2010, 2015, 2020  
*metadata*: [NA_landcover_2020v2_30m_metadata.html](metadata/NA_landcover_2020v2_30m_metadata.html)  
*how to cite*:  
Commission for Environmental Cooperation (CEC). 2024. "North American Environmental Atlas - Land Cover 2020 30m". North American Land Change Monitoring System. Canada Centre for Remote Sensing (CCRS), U.S. Geological Survey (USGS), Comisión Nacional para el Conocimiento y Uso de la Biodiversidad (CONABIO), Comisión Nacional Forestal (CONAFOR), Instituto Nacional de Estadística y Geografía (INEGI). Ed. 2.0, Raster digital data [30-m]. Available at https://www.cec.org/north-american-environmental-atlas/land-cover-30m-2020/

### Wetlands

##### [National Wetlands Inventory](https://www.fws.gov/program/national-wetlands-inventory)
*source*: United States Fish and Wildlife Service (USFWS)  
*years*: continually updated  
*metadata*: https://documentst.ecosphere.fws.gov/wetlands/data/metadata/FWS_Wetlands.xml  
*how to cite*:  
U. S. Fish and Wildlife Service. May 1st, 2024. National Wetlands Inventory website. U.S. Department of the Interior, Fish and Wildlife Service, Washington, D.C.

### Bird Count Locations

Bird count locations were provided by Annie & Isabel. From the original methods notes:
- Great Lakes Coastal Wetland Monitoring Program (CWMP) bird point count locations (PCL): [CWMP_PCL_all.xlsx](https://docs.google.com/spreadsheets/d/1-FZrtUYZS9HEFVKIO94fKBhGXi5KQDon)
- GLFWRA bird point count locations (PCL): GLFWRA_PCL_all.csv


In [ ]:
# Weird proj Issue
# 0) RESTART the Jupyter kernel first (not just "Run All").
# 1) Run this as the very first cell:
import os, sys

# Drop any OSGeo4W bleed-through
for k in ["PROJ_LIB", "PROJ_DATA", "GDAL_DATA"]:
    if os.environ.get(k, "").lower().find("osgeo4w") != -1:
        os.environ.pop(k, None)
os.environ["PATH"] = ";".join(
    p for p in os.environ["PATH"].split(";") if "OSGeo4W" not in p
)

from pyproj import CRS, datadir

crs4326 = CRS.from_epsg(4326)
print("CRS ok:", crs4326)
print("pyproj datadir:", datadir.get_data_dir())
print("PROJ_LIB:", os.environ.get("PROJ_LIB"))

In [ ]:
# IMPORTS

# external
import geopandas as gpd
import pandas as pd
# local
from libs.land_cover import nlcd_sesh_setup, process_one_landcover
from libs.nwi import process_one_nwi

In [ ]:
# CONSTANTS

# buffer sizes
buffers = [200, 500, 1000, 2000]

# years
years = [2010, 2015, 2020, 2023]

# project CRS
crs = "EPSG:5070"

In [ ]:
# Paths

# cwmp bird count locations
cwmp_path = "data/CWMP_PCL_all.xlsx - CWMP_bird_point_count_locations.csv"

# nalcms land cover paths
nalcms_paths = {
    2010: r"M:\isabel_bird\data\nalcms\land_cover_2010v3_30m_tif\NA_NALCMS_landcover_2010v3_30m\data\NA_NALCMS_landcover_2010v3_30m.tif",
    2015: r"M:\isabel_bird\data\nalcms\land_cover_2015v4_30m_tif\NA_NALCMS_landcover_2015v4_30m\data\NA_NALCMS_landcover_2015v4_30m.tif",
    2020: r"M:\isabel_bird\data\nalcms\land_cover_2020v2_30m_tif\NA_NALCMS_landcover_2020v2_30m\data\NA_NALCMS_landcover_2020v2_30m.tif",
}

# nwi data directory
nwi_dir = r"M:\isabel_bird\data\nwi"

## Load bird count locations

- [coastal wetlands](https://docs.google.com/spreadsheets/d/1-FZrtUYZS9HEFVKIO94fKBhGXi5KQDon)
  - field mapping
    - lat: latitude
    - lon: longitude
- GLFWRA


In [ ]:
# load cwmp table
cwmp_df = pd.read_csv(cwmp_path)
print(f"CWMP bird count locations: {len(cwmp_df)}")
cwmp_df.head()

## Process each location, buffer

1. create buffer
2. read in raster for buffer
3. calculate stats for buffer

- for each class:
  - count
  - pct of total


### dev loop

In [ ]:
%%script false
# store results
results = []

# subset for testing
df = cwmp_df.head()

# Iterate over CWMP locations
for index, row in df.iterrows():
    print(f"Processing location {index + 1} of {len(df)}")
    # create point
    point = gpd.points_from_xy(
        [row["longitude"]], [row["latitude"]], crs="EPSG:4326"
    )

    # iterate over buffer distances
    for buffer in buffers[:1]:
        print(f"Processing buffer {buffer}m")
        # iterate over years
        for year in years[:1]:
            print(f"Processing year {year}")

            # process nlcd data for row and append to results
            results.append(process_one_landcover(point, buffer, year, "nlcd"))

            # process nalcms data for row and append to results
            results.append(process_one_landcover(point, buffer, year, "nalcms"))

            # process nwi data for row and append to results
            results.append(process_one_nwi(point, buffer, "nwi"))

            # process a row and plot it
            # process_one_for_show(row, buffer, year)

# merge all dicts in `results` into one flat dict
combined = {k: v for d in results for k, v in d.items()}

# now this matches the single-row index of df
new_cols = pd.DataFrame([combined], index=df.index)
results_df = df.join(new_cols)
display(results_df)

### optimized loop

In [ ]:
%%script false
from tqdm.auto import tqdm

rows_out = []  # one dict per input row
df = cwmp_df.head()  # full dataframe
loc_iter = df.itertuples(index=True)  # keep original index
for row in tqdm(
    loc_iter,
    total=len(df),
    desc="Locations",
    position=0,
    dynamic_ncols=True,
):
    row_results = []  # collect dicts for this row
    point = gpd.points_from_xy([row.longitude], [row.latitude], crs="EPSG:4326")

    with tqdm(
        total=len(buffers), desc="Buffers", leave=False, position=1
    ) as p_buf:
        for b_idx, buffer in enumerate(buffers, 1):
            p_buf.set_postfix(buffer=buffer, b_num=b_idx)

            with tqdm(
                total=len(years), desc="Years", leave=False, position=2
            ) as p_year:
                for y_idx, year in enumerate(years, 1):
                    # show both buffer + year on the inner bar
                    p_year.set_postfix(buffer=buffer, year=year, y_num=y_idx)
                    row_results.append(
                        process_one_landcover(point, buffer, year, "nlcd")
                    )
                    row_results.append(
                        process_one_landcover(point, buffer, year, "nalcms")
                    )
                    row_results.append(process_one_nwi(point, buffer, "nwi"))
                    
                    p_year.update(1)
            p_buf.update(1)

    # merge this location's dicts into one dict
    merged = {k: v for d in row_results for k, v in d.items()}
    merged["_source_index"] = row.Index  # keep the original row index
    rows_out.append(merged)

# build a DataFrame aligned to original df
new_cols = pd.DataFrame(rows_out).set_index("_source_index").reindex(df.index)
results_df = df.join(new_cols)

display(results_df.head())

### parallelized

In [ ]:
# Robust, notebook/Windows-safe parallelism with joblib (loky) + tqdm,
# avoiding closures and unpicklable objects. Adds per-call diagnostics.

import os, time, traceback
from typing import List, Dict, Tuple, Optional
import pandas as pd
import geopandas as gpd
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# ── limit nested threading to prevent oversubscription ──
os.environ.setdefault("GDAL_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# ── tqdm <-> joblib bridge without closures ──
from joblib.parallel import BatchCompletionCallBack


class _TqdmBatchCompletionCallback(BatchCompletionCallBack):
    def __init__(self, *args, **kwargs):
        self._pbar = kwargs.pop("pbar", None)
        super().__init__(*args, **kwargs)

    def __call__(self, *args, **kwargs):
        if self._pbar is not None:
            self._pbar.update(n=self.batch_size)
        return super().__call__(*args, **kwargs)


def _monkeypatch_tqdm_joblib(pbar):
    """Context manager to show joblib progress in tqdm without capturing closures."""
    from contextlib import contextmanager

    @contextmanager
    def _ctx():
        orig = Parallel._backend_callback

        def _factory(*a, **k):
            k["pbar"] = pbar
            return _TqdmBatchCompletionCallback(*a, **k)

        Parallel._backend_callback = _factory
        try:
            yield
        finally:
            Parallel._backend_callback = orig

    return _ctx()


# ── minimal row iterator ──
def _iter_min_rows(df: pd.DataFrame):
    for r in df.itertuples(index=True):
        yield int(r.Index), float(r.longitude), float(r.latitude)


# ── single-call wrapper with timing + error capture (NO closures) ──
def _call_with_diag(
    tag: str, func_name: str, *args, **kwargs
) -> Tuple[Optional[dict], Dict]:
    """
    func_name: name of a global callable available in the worker's module scope
               (e.g., 'process_one_landcover' or 'process_one_nwi')
    """
    t0 = time.time()
    ok, err, tb, out = True, None, None, None
    try:
        fn = globals()[
            func_name
        ]  # resolve in child without pickling the callable
        out = fn(*args, **kwargs)
    except Exception as e:
        ok, err, tb = False, repr(e), traceback.format_exc(limit=8)
    diag = {
        "tag": tag,
        "ok": ok,
        "elapsed_sec": round(time.time() - t0, 4),
        "error": err,
        "traceback": tb,
    }
    return (out if ok else None, diag)


# ── top-level worker (importable, no captured state) ──
def compute_one_row_instrumented(
    row_index: int, lon: float, lat: float, buffers: List, years: List
) -> Dict:
    point = gpd.points_from_xy([lon], [lat], crs="EPSG:4326")

    merged_parts: List[dict] = []
    diags: List[Dict] = []

    for buffer in buffers:
        for year in years:
            # NLCD
            res, d = _call_with_diag(
                f"nlcd|b={buffer}|y={year}",
                "process_one_landcover",
                point,
                buffer,
                year,
                "nlcd",
            )
            d["row_index"] = row_index
            diags.append(d)
            if res is not None:
                merged_parts.append(res)

            # NALCMS
            res, d = _call_with_diag(
                f"nalcms|b={buffer}|y={year}",
                "process_one_landcover",
                point,
                buffer,
                year,
                "nalcms",
            )
            d["row_index"] = row_index
            diags.append(d)
            if res is not None:
                merged_parts.append(res)

            # NWI
            res, d = _call_with_diag(
                f"nwi|b={buffer}", "process_one_nwi", point, buffer, "nwi"
            )
            d["row_index"] = row_index
            diags.append(d)
            if res is not None:
                merged_parts.append(res)

    merged = {k: v for dct in merged_parts for k, v in dct.items()}
    merged["_source_index"] = row_index
    return {"merged": merged, "diags": diags}


# ── main driver ──
def run_parallel_locations_joblib_instrumented(
    df: pd.DataFrame,
    buffers: List,
    years: List,
    n_jobs: int = 4,
    diag_csv: Optional[str] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    jobs = list(_iter_min_rows(df))
    if not jobs:
        return df.copy(), pd.DataFrame(
            columns=[
                "row_index",
                "tag",
                "ok",
                "elapsed_sec",
                "error",
                "traceback",
            ]
        )

    # batch_size=1 → steady progress + early failure surfacing
    with tqdm(
        total=len(jobs), desc="Locations (done)", dynamic_ncols=True
    ) as pbar, _monkeypatch_tqdm_joblib(pbar):
        results = Parallel(
            n_jobs=n_jobs,
            prefer="processes",
            batch_size=1,
            verbose=0,
        )(
            [
                delayed(compute_one_row_instrumented)(
                    idx, lon, lat, buffers, years
                )
                for (idx, lon, lat) in jobs
            ]
        )

    merged_rows = [r["merged"] for r in results]
    all_diags = [d for r in results for d in r["diags"]]

    new_cols = (
        pd.DataFrame(merged_rows).set_index("_source_index").reindex(df.index)
    )
    results_df = df.join(new_cols)

    diag_df = pd.DataFrame(all_diags)
    if diag_csv:
        diag_df.to_csv(diag_csv, index=False)

    return results_df, diag_df


# ── usage ──
df = cwmp_df.head()
results_df, diag_df = run_parallel_locations_joblib_instrumented(
    df, buffers=buffers, years=years, n_jobs=4, diag_csv="diag_calls.csv"
)
display(results_df.head())
display(diag_df.head())

In [ ]:
# in your current notebook
from joblib import Parallel, delayed
import time

res = Parallel(n_jobs=4, prefer="processes")(
    delayed(lambda i: (i, time.sleep(0.2)))(i) for i in range(10)
)
print("ok", len(res))

In [ ]:
# export results to csv
outpath = r"C:\Projects\isabel_bird\data\output\cwmp_bird_count_landcover_and_wetland_summary.csv"
results_df.to_csv(outpath, index=False)

## Process One Loop - Plotting Output

Modify the code in the selected cell to create a map showing the buffer outline overlaid on the clipped raster with distinct colors for categorical data, create a dataframe combining the class counts and class percent dictionaries, and create a pie chart plot based on the class count percent data.


### Install necessary libraries

#### Subtask:

Install libraries for plotting and data manipulation, such as `matplotlib` and `seaborn`.


**Reasoning**:
Install libraries for plotting and data manipulation, such as `matplotlib` and `seaborn`.


In [ ]:
%pip install matplotlib seaborn
import matplotlib.pyplot as plt
import seaborn as sns

### `process one for show` function

#### Subtask:

Update the function to perform the following tasks:

- Create a map showing the buffer outline overlaid on the clipped raster with distinct colors for categorical data.
- Create a pandas DataFrame combining the class counts and class percent dictionaries.
- Generate a pie chart based on the class percent data.


In [ ]:
import rasterio.plot
import rasterio.windows
import math  # Import math for isnan check
import geopandas as gpd  # Import geopandas


def process_one_for_show(row: pd.Series, buffer_size: int, year: int):
    # create point
    point = gpd.points_from_xy(
        [row["longitude"]], [row["latitude"]], crs="EPSG:4326"
    )

    # Check if point creation was successful and coordinates are valid
    if point.is_empty or not all(math.isfinite(c) for c in point[0].coords[0]):
        print(
            f"Skipping location {row['site_id']} due to invalid coordinates: {row['longitude']}, {row['latitude']}"
        )
        return None

    # get raster values for buffer
    with rasterio.Env(aws_sesh):
        with rasterio.open(nlcd_paths[year]) as src:
            # reproject point to raster CRS - so units match buffer units
            point_reproj = point.to_crs(src.crs)
            # create buffer from point, convert to geojson like object
            buffer_reproj = point_reproj.buffer(buffer_size)

            # Check if buffer creation was successful and bounds are valid
            # Iterate over the array within the list returned by buffer_reproj.bounds
            if buffer_reproj.is_empty or (
                buffer_reproj.bounds is not None
                and any(math.isnan(c) for c in buffer_reproj.bounds[0])
            ):
                print(
                    f"Skipping location {row['site_id']} due to invalid buffer geometry. Bounds: {buffer_reproj.bounds}"
                )
                return None

            # Add this check to see what buffer_reproj.bounds contains
            print(f"buffer_reproj.bounds: {buffer_reproj.bounds}")

            # read data from raster within poly (window)
            data, transform = mask(
                src, buffer_reproj, crop=True, filled=True
            )  # windowed read
            # get values from first band, skipping nodata values
            band = data[0]  # NLCD is single-band categorical
            nodata = src.nodatavals[0]
            mask_valid = (
                (band != nodata)
                if nodata is not None
                else np.ones_like(band, dtype=bool)
            )
            vals = band[mask_valid].astype(np.int8)

            # get count by class
            unique_values, counts = np.unique(vals, return_counts=True)
            class_counts = dict(zip(unique_values, counts))
            # print(f"class counts: {class_counts}")

            # calculate percent of total area occupied by each class
            total = np.sum(counts)
            class_percents = {k: v / total for k, v in class_counts.items()}
            print(f"class percents: {class_percents}")

            # 2. Create a colormap for the raster data.
            # Filter out nodata values from unique_values for colormap
            unique_values_filtered = [
                val
                for val in unique_values
                if (nodata is None or val != nodata)
            ]
            num_classes = len(unique_values_filtered)
            colors = plt.cm.tab20(
                np.linspace(0, 1, num_classes)
            )  # Use a colormap with enough distinct colors
            cmap = plt.cm.colors.ListedColormap(colors)
            bounds = (
                np.array(
                    unique_values_filtered + [unique_values_filtered[-1] + 1]
                )
                - 0.5
            )
            norm = plt.cm.colors.BoundaryNorm(bounds, cmap.N)

            # 3. Plot the clipped raster data
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
            # Pass the buffer bounds directly to the extent argument of rasterio.plot.show
            extent = buffer_reproj.bounds[0]
            rasterio.plot.show(
                band,
                transform=transform,
                ax=ax1,
                cmap=cmap,
                norm=norm,
                extent=extent,
            )

            # 4. Plot the buffer geometry - convert to GeoSeries for plotting
            gpd.GeoSeries(buffer_reproj).plot(
                ax=ax1, facecolor="none", edgecolor="red", linewidth=2
            )

            # 5. Add a title to the plot
            ax1.set_title(
                f"Land Cover for Location {row['site_id']}, Buffer {buffer_size}m, Year {year}"
            )

            # 6. Create a pandas DataFrame from the class_counts and class_percents dictionaries.
            results_df = pd.DataFrame(
                {
                    "count": pd.Series(class_counts),
                    "percent": pd.Series(class_percents),
                }
            )

            # 7. Add a column to the DataFrame for the NLCD class names
            results_df["class_name"] = results_df.index.map(NLCD_CLASS_LUT)

            # 8. Generate a pie chart
            ax2.pie(
                results_df["percent"],
                labels=results_df["class_name"],
                autopct="%1.1f%%",
                startangle=90,
                colors=colors[: len(results_df)],
            )

            # 9. Add a title to the pie chart.
            ax2.set_title(
                f"Land Cover Distribution for Location {row['site_id']}, Buffer {buffer_size}m, Year {year}"
            )

            # 10. Ensure both plots are displayed
            plt.show()

    return None